# AI-Based Intelligent Intrusion Detection System (AI-IDS)
## Machine Learning Experimentation & Academic Benchmark Notebook
**Project Title:** Development of an AI-Based Intelligent Intrusion Detection System for Advanced Cybersecurity Applications

### Pipeline Overview:
1. **Data Ingestion**: NSL-KDD Network Intrusion Dataset
2. **Data Preprocessing**: Categorical Encoding (`protocol_type`, `service`, `flag`) & Standard Scaling
3. **Feature Selection**: SelectKBest ($f\_classif$ ANOVA)
4. **Dimensionality Reduction**: Principal Component Analysis (PCA)
5. **Stratified 70:30 Split**: Training and Testing partitions
6. **Model Training & Evaluation**:
   - Supervised: Random Forest Classifier
   - Supervised: Support Vector Machine (SVM)
   - Unsupervised: K-Means Centroid Anomaly Clustering
   - Unsupervised: Isolation Forest Anomaly Outliers

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("All ML libraries successfully imported.")

### 1. Ingest Dataset & Attack Categorization

In [ ]:
data_path = '../data/sample.csv'
df = pd.read_csv(data_path)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} features.")
df.head()

### 2. Preprocessing & Feature Selection

In [ ]:
categorical_cols = ['protocol_type', 'service', 'flag']
drop_cols = ['label', 'difficulty_level', 'class'] if 'label' in df.columns else []
feature_cols = [c for c in df.columns if c not in drop_cols]
numerical_cols = [c for c in feature_cols if c not in categorical_cols]

# Target binary: 0 = Normal, 1 = Attack
y = (df['label'] != 'normal').astype(int)
X = df[feature_cols]

# 70:30 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Feature Selection (SelectKBest K=20)
selector = SelectKBest(score_func=f_classif, k=min(20, X_train_proc.shape[1]))
X_train_sel = selector.fit_transform(X_train_proc, y_train)
X_test_sel = selector.transform(X_test_proc)

# PCA Reduction (Components=10)
pca = PCA(n_components=min(10, X_train_sel.shape[1]), random_state=42)
X_train_pca = pca.fit_transform(X_train_sel)
X_test_pca = pca.transform(X_test_sel)
print(f"PCA Total Explained Variance: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

### 3. Model Training: Random Forest, SVM & K-Means

In [ ]:
# 1. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sel, y_train)
rf_preds = rf.predict(X_test_sel)

# 2. Support Vector Machine (SVM)
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_pca, y_train)
svm_preds = svm.predict(X_test_pca)

# 3. K-Means Anomaly Detector
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_train_pca)
train_dist = np.linalg.norm(X_train_pca - kmeans.cluster_centers_[kmeans.labels_], axis=1)
thresh = np.percentile(train_dist, 95.0)
test_dist = np.linalg.norm(X_test_pca - kmeans.cluster_centers_[kmeans.predict(X_test_pca)], axis=1)
kmeans_preds = (test_dist > thresh).astype(int)

print("Model training completed.")

### 4. Model Comparative Benchmarks & Metrics Calculation

In [ ]:
def evaluate(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
    return {
        'Model': name,
        'Accuracy (%)': round(acc, 2),
        'Precision (%)': round(prec, 2),
        'Recall (%)': round(rec, 2),
        'F1 Score (%)': round(f1, 2),
        'FPR (%)': round(fpr, 2)
    }

results = [
    evaluate('Random Forest', y_test, rf_preds),
    evaluate('Support Vector Machine (SVM)', y_test, svm_preds),
    evaluate('K-Means (Unsupervised Distance)', y_test, kmeans_preds)
]

results_df = pd.DataFrame(results)
results_df